In [1]:
import os
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split
from collections import Counter

# 1. Generate Realistic Customer Reviews Data
np.random.seed(42)
n_samples = 2000

pos_reviews = [
    "great product loved it", "excellent quality highly recommend", "amazing experience working fine",
    "best purchase ever fast delivery", "very good service satisfied", "awesome item perfect fit",
    "love this so much brilliant", "highly efficient works great", "fantastic quality absolutely beautiful"
]
neg_reviews = [
    "terrible product bad quality", "worst experience do not buy", "broken on arrival useless",
    "waste of money very disappointed", "poor service slow delivery", "not working stopped working",
    "hate this item horrible", "bad customer service useless item", "defective piece completely broken"
]

reviews, sentiments = [], []
for _ in range(n_samples):
    if np.random.rand() > 0.5:
        reviews.append(np.random.choice(pos_reviews))
        sentiments.append(1) # Positive
    else:
        reviews.append(np.random.choice(neg_reviews))
        sentiments.append(0) # Negative

# 2. Save into CSV Format
data_dir = r"D:\DL_PROJECTS\Project5_Sentiment_RNN\data"
os.makedirs(data_dir, exist_ok=True)
csv_path = os.path.join(data_dir, "customer_reviews.csv")

df = pd.DataFrame({'review_text': reviews, 'sentiment': sentiments})
df.to_csv(csv_path, index=False)

print(f"✅ NLP Sentiment CSV Created Successfully at: {csv_path}")
print(f"Dataset Shape: {df.shape}")
print(df.head())

✅ NLP Sentiment CSV Created Successfully at: D:\DL_PROJECTS\Project5_Sentiment_RNN\data\customer_reviews.csv
Dataset Shape: (2000, 2)
                         review_text  sentiment
0  bad customer service useless item          0
1        love this so much brilliant          1
2            hate this item horrible          0
3  bad customer service useless item          0
4  bad customer service useless item          0


In [ ]:
# 1. Vocabulary (Word Dictionary) Banayein
all_text = " ".join(df['review_text'].values).split()
word_counts = Counter(all_text)
# Sort words by frequency and create mapping (0 is reserved for padding)
vocab = {word: i+1 for i, (word, _) in enumerate(word_counts.items())}
vocab_size = len(vocab) + 1 # Plus 1 for padding token 0

# 2. Text ko Numbers (Sequences) mein badlein
sequences = []
for review in df['review_text'].values:
    seq = [vocab[word] for word in review.split() if word in vocab]
    sequences.append(seq)

# 3. Padding (Har review ko maximum 5 words ka banayein)
max_len = 5
padded_sequences = np.zeros((len(sequences), max_len), dtype=int)
for i, seq in enumerate(sequences):
    if len(seq) <= max_len:
        padded_sequences[i, :len(seq)] = seq
    else:
        padded_sequences[i, :] = seq[:max_len]

# 4. Train Test Split & PyTorch Loaders

X = padded_sequences
y = df['sentiment'].values

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

X_train_t = torch.tensor(X_train, dtype=torch.long)
y_train_t = torch.tensor(y_train, dtype=torch.float32).unsqueeze(1)
X_test_t = torch.tensor(X_test, dtype=torch.long)
y_test_t = torch.tensor(y_test, dtype=torch.float32).unsqueeze(1)

train_dataset = TensorDataset(X_train_t, y_train_t)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)

print(f"Vocabulary Size: {vocab_size}")
print(f"Padded X_train shape: {X_train_t.shape} -> [Samples, Max_Words]")

Vocabulary Size: 59
Padded X_train shape: torch.Size([1600, 5]) -> [Samples, Max_Words]


In [3]:
class SentimentRNN(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim):
        super(SentimentRNN, self).__init__()
        # Embedding Layer: Words ko dense vector mein convert karti hai
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        # GRU Layer: Advanced Recurrent Neural Network
        self.gru = nn.GRU(embed_dim, hidden_dim, batch_first=True)
        # Fully Connected Output
        self.fc = nn.Linear(hidden_dim, 1)
        self.sigmoid = nn.Sigmoid()
        
    def forward(self, x):
        embedded = self.embedding(x)
        # GRU output and hidden state
        _, h_n = self.gru(embedded) # h_n shape: [1, batch_size, hidden_dim]
        
        # Take the last hidden state output
        out = self.fc(h_n.squeeze(0))
        out = self.sigmoid(out)
        return out

# Initialize Model (Embedding Dim = 16, Hidden Units = 32)
model = SentimentRNN(vocab_size=vocab_size, embed_dim=16, hidden_dim=32)
print(model)

SentimentRNN(
  (embedding): Embedding(59, 16, padding_idx=0)
  (gru): GRU(16, 32, batch_first=True)
  (fc): Linear(in_features=32, out_features=1, bias=True)
  (sigmoid): Sigmoid()
)


In [6]:
criterion = nn.BCELoss()
optimizer = optim.Adam(model.parameters(), lr=0.005)

epochs = 20
print("Starting Text-RNN Sentiment Model Training...")

for epoch in range(epochs):
    model.train()
    epoch_loss = 0.0
    
    for batch_X, batch_y in train_loader:
        outputs = model(batch_X)
        loss = criterion(outputs, batch_y)
        
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        epoch_loss += loss.item()
        
    print(f"Epoch [{epoch+1}/{epochs}] -> Training Loss: {epoch_loss/len(train_loader):.4f}")

print("🎉 Text Model Trained Successfully!")

Starting Text-RNN Sentiment Model Training...


Epoch [1/20] -> Training Loss: 0.0000
Epoch [2/20] -> Training Loss: 0.0000
Epoch [3/20] -> Training Loss: 0.0000
Epoch [4/20] -> Training Loss: 0.0000
Epoch [5/20] -> Training Loss: 0.0000
Epoch [6/20] -> Training Loss: 0.0000
Epoch [7/20] -> Training Loss: 0.0000
Epoch [8/20] -> Training Loss: 0.0000
Epoch [9/20] -> Training Loss: 0.0000
Epoch [10/20] -> Training Loss: 0.0000
Epoch [11/20] -> Training Loss: 0.0000
Epoch [12/20] -> Training Loss: 0.0000
Epoch [13/20] -> Training Loss: 0.0000
Epoch [14/20] -> Training Loss: 0.0000
Epoch [15/20] -> Training Loss: 0.0000
Epoch [16/20] -> Training Loss: 0.0000
Epoch [17/20] -> Training Loss: 0.0000
Epoch [18/20] -> Training Loss: 0.0000
Epoch [19/20] -> Training Loss: 0.0000
Epoch [20/20] -> Training Loss: 0.0000
🎉 Text Model Trained Successfully!


In [5]:
model.eval()
with torch.no_grad():
    test_preds = model(X_test_t)
    predicted_classes = (test_preds >= 0.5).float()
    correct = (predicted_classes == y_test_t).sum().item()
    accuracy = (correct / y_test_t.size(0)) * 100

print(f"🔥 Final Sentiment Analysis Accuracy: {accuracy:.2f}%\n")

# Custom Live Testing Function
def predict_custom_review(review_sentence):
    model.eval()
    # Tokenize and Pad custom review
    words = review_sentence.lower().split()
    tokens = [vocab[w] for w in words if w in vocab]
    
    padded = np.zeros((1, max_len), dtype=int)
    if len(tokens) <= max_len:
        padded[0, :len(tokens)] = tokens
    else:
        padded[0, :] = tokens[:max_len]
        
    tensor_in = torch.tensor(padded, dtype=torch.long)
    with torch.no_grad():
        prob = model(tensor_in).item()
    
    sentiment = "😊 Positive" if prob >= 0.5 else "😡 Negative"
    print(f"Review: '{review_sentence}' -> Predicted Sentiment: {sentiment} (Confidence: {prob*100:.1f}%)")

# Live Tests
predict_custom_review("great product loved it")
predict_custom_review("worst experience do not buy")

🔥 Final Sentiment Analysis Accuracy: 100.00%

Review: 'great product loved it' -> Predicted Sentiment: 😊 Positive (Confidence: 100.0%)
Review: 'worst experience do not buy' -> Predicted Sentiment: 😡 Negative (Confidence: 0.0%)
